In [2]:
import pandas as pd

# 1. 방금 만든 최종 마스터 데이터 불러오기
df = pd.read_csv('대청호_AI_학습용_최종_마스터데이터.csv')

print("🛠️ 피처 엔지니어링 2.0 (도메인 지식 + 데이터 누수 방지) 시작...")

# -------------------------------------------------------------
# ① 데이터 정렬 (Data Sorting)
# -------------------------------------------------------------
df['조사일'] = pd.to_datetime(df['조사일'])
df = df.sort_values(by=['채수위치', '조사일']).reset_index(drop=True)

# -------------------------------------------------------------
# ② 데이터 누수(Data Leakage) 원천 차단 (정답지 찢어버리기)
# -------------------------------------------------------------
# 컬럼명에 '우점종', '냄새', '조류독소'가 들어간 열은 전부 압수!
leakage_cols = [col for col in df.columns if '우점종' in col or '냄새' in col or '독소' in col]
df = df.drop(columns=leakage_cols)
print(f"✅ 데이터 누수 변수 {len(leakage_cols)}개 제거 완료")

# -------------------------------------------------------------
# ③ 타임머신 (7일 전 데이터로 밀기 - Lag Feature)
# -------------------------------------------------------------
# 우리가 7일 뒤(오늘)의 남조류를 맞추기 위해 써야 할 '과거의 원인' 변수들
features_to_shift = [
    '수온(℃)', 'pH', 'DO(㎎/L)', '투명도', '탁도', 'Chl-a (㎎/㎥)', 
    '평균기온(°C)', '일강수량(mm)', '합계 일조시간(hr)', '합계 일사량(MJ/m2)',
    'TN(㎎/L)', 'TP(㎎/L)', 'N_P_비율', '체류시간_지표', 
    '유해남조류 세포수 (cells/㎖)' # 7일 전 남조류 수치(씨앗)도 엄청 중요!
]

# 채수위치(문의/추동/회남)가 섞이지 않게 그룹별로 1칸(7일)씩 끌어내림
for feature in features_to_shift:
    df[f'7일 전_{feature}'] = df.groupby('채수위치')[feature].shift(1)

# -------------------------------------------------------------
# ④ 쓸모없는 현재 데이터 & 노이즈 제거
# -------------------------------------------------------------
# 예측 시점(D-7)에서는 '오늘의 수온'을 알 수 없으므로 현재 원인 변수들은 다 지움!
# (🚨 단, 우리가 맞춰야 할 정답인 '현재 유해남조류 세포수'는 절대 지우면 안 됨!!)
features_to_drop = [f for f in features_to_shift if f != '유해남조류 세포수 (cells/㎖)']
df = df.drop(columns=features_to_drop)

# 중요도가 1% 미만이었던 '채수위치'와 단순 텍스트 '분류', '지점명' 제거
df = df.drop(columns=['분류', '지점명', '채수위치', '조사일', '수위(EL.m)', '저수량(백만㎥)', '저수율(%)', '강우량(mm)', '유입량(㎥/s)', '총방류량(㎥/s)'])

# shift 하면서 생긴 첫 줄의 결측치(NaN)들 깔끔하게 삭제
df = df.dropna()

# -------------------------------------------------------------
# ⑤ 찐 최종 모델링용 데이터 저장
# -------------------------------------------------------------
df.to_csv('대청호_ML_학습용_최종_전처리완료.csv', index=False, encoding='utf-8-sig')

print(f"✅ 피처 엔지니어링 완료! 최종 데이터 형태: {df.shape}")
print("🚀 이제 진짜 AI 모델링(Random Forest)에 들어갈 준비가 끝났습니다!")

🛠️ 피처 엔지니어링 2.0 (도메인 지식 + 데이터 누수 방지) 시작...
✅ 데이터 누수 변수 12개 제거 완료
✅ 피처 엔지니어링 완료! 최종 데이터 형태: (584, 16)
🚀 이제 진짜 AI 모델링(Random Forest)에 들어갈 준비가 끝났습니다!
